# IT594 Project: Notebook 06, an independent check of the reranker

The reranker (notebook 04) and the violation grader both read the product's **color field**, so the measured drop in
violations is partly built in. This notebook judges the reranker with evidence that **never uses the color field**:

1. **Blind check (main):** a random sample of products from the top 10 of `hybrid` and `hybrid +rerank`, shuffled, with the
   system hidden and the color field removed. Each product is labeled from its **name and description only**.
2. **Name-only grader (secondary):** an automatic check that reads only the product name. It can judge few products
   (about 7% of names state a color) and favours BM25, so it is a cross-check, not the main evidence.

The reranker is run with penalty ∞ (the value cross-validation chose on 5 of 6 halves in notebook 04).

## 1. Setup, runs and the reranker

In [1]:
import os, re, sys
import numpy as np, pandas as pd

sys.path.insert(0, "../src")
from metrics import ndcg_at_k, constraint_violation_rate, paired_bootstrap
from color_checker import query_colors, color_check, parse_colors
from retrieval import rrf
from rerank import field_violates, demote

PATH = "../data/WANDS/dataset/"
products = pd.read_csv(PATH + "product.csv", sep="\t")
queries  = pd.read_csv(PATH + "query.csv",   sep="\t")
labels   = pd.read_csv(PATH + "label.csv",   sep="\t")
labels["gain"] = labels["label"].map({"Exact": 2, "Partial": 1, "Irrelevant": 0})
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in labels.groupby("query_id")}
qtext = dict(zip(queries["query_id"], queries["query"]))
all_q = list(queries["query_id"])

def parse_features(s):
    d = {}
    if pd.isna(s):
        return d
    for pair in s.split("|"):
        if ":" in pair:
            k, v = pair.split(":", 1)
            d[k.strip().lower()] = v.strip()
    return d

feat = products["product_features"].apply(parse_features)
pcolor = dict(zip(products["product_id"], feat.apply(lambda d: d.get("color") or d.get("primarycolor") or "")))
for c in ["product_name", "product_class", "product_description"]:
    products[c] = products[c].fillna("")
pname = dict(zip(products["product_id"], products["product_name"]))
pdesc = dict(zip(products["product_id"], products["product_description"]))
pids = products["product_id"].values
text = products["product_name"] + ". " + products["product_class"] + ". " + products["product_description"].str.slice(0, 500)

In [2]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([tok(t) for t in text])
runs = {"bm25": {q: list(pids[np.argsort(-bm25.get_scores(tok(qtext[q])))[:100]]) for q in all_q}}

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
EMB = "wands_emb.npy"     # same cache file as notebooks 03-05 and eval
if os.path.exists(EMB):
    E = np.load(EMB)
else:
    E = model.encode(text.tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(EMB, E)
S = model.encode([qtext[q] for q in all_q], normalize_embeddings=True) @ E.T
runs["dense"] = {q: list(pids[np.argsort(-S[i])[:100]]) for i, q in enumerate(all_q)}
runs["hybrid_rrf"] = {q: rrf([runs["bm25"][q], runs["dense"][q]]) for q in all_q}

K, LENIENT = 10, True
color_qids = [q for q in all_q if query_colors(qtext[q], LENIENT)]

def signal(qid):
    qcols = query_colors(qtext[qid], LENIENT)
    return lambda p: field_violates(qcols, pcolor.get(p, ""), LENIENT)

for n in ["bm25", "dense", "hybrid_rrf"]:
    runs[n + " +rerank"] = {**runs[n], **{q: demote(runs[n][q], signal(q), float("inf")) for q in color_qids}}
print("color queries:", len(color_qids), "| systems:", list(runs))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

color queries: 44 | systems: ['bm25', 'dense', 'hybrid_rrf', 'bm25 +rerank', 'dense +rerank', 'hybrid_rrf +rerank']


## 2. Secondary check: a grader that reads only the product name

Same outcomes as the main checker, but the color field is never read: a product is checkable only if its **name** states a color.

In [3]:
def name_grader(qid):
    qcols = query_colors(qtext[qid], LENIENT)
    def g(p):
        cols, multi = parse_colors(pname.get(p, ""), LENIENT, False)
        if not cols:
            return "unknown"
        return "match" if qcols & cols else ("ambiguous" if multi else "mismatch")
    return g

def field_grader(qid):
    qcols = query_colors(qtext[qid], LENIENT)
    return lambda p: color_check(qcols, pcolor.get(p, ""), pname.get(p, ""), LENIENT)

def viol(run, grader, among="checkable"):
    return [constraint_violation_rate(run[q], grader(q), K, among) for q in color_qids]

def coverage(run, grader):
    return np.mean([np.mean([grader(q)(p) in ("match", "mismatch") for p in run[q][:K]]) for q in color_qids])

table = pd.DataFrame({n: {"field grader: violation": np.nanmean(viol(runs[n], field_grader)),
                          "name-only grader: violation": np.nanmean(viol(runs[n], name_grader)),
                          "name-only grader: checkable share": coverage(runs[n], name_grader)}
                      for n in runs}).T
display(table.round(3))

for n in ["bm25", "dense", "hybrid_rrf"]:
    m, (lo, hi), k = paired_bootstrap(viol(runs[n + " +rerank"], name_grader), viol(runs[n], name_grader))
    print(f"name-only grader, {n} +rerank - {n}: {m:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (n={k} queries)")

,field grader: violation,name-only grader: violation,name-only grader: checkable share
bm25,0.316,0.156,0.361
dense,0.592,0.299,0.325
hybrid_rrf,0.416,0.195,0.359
bm25 +rerank,0.065,0.090,0.368
dense +rerank,0.166,0.214,0.302
hybrid_rrf +rerank,0.134,0.171,0.357


name-only grader, bm25 +rerank - bm25: -0.046  95% CI [-0.093, -0.006]  (n=34 queries)
name-only grader, dense +rerank - dense: -0.088  95% CI [-0.185, -0.005]  (n=29 queries)
name-only grader, hybrid_rrf +rerank - hybrid_rrf: -0.007  95% CI [-0.055, +0.041]  (n=34 queries)


**Reading it.** If the reranker only "agreed with the field", the name-only grader would show no improvement. A drop under the
name-only grader is evidence from a source the reranker never reads. Few products are checkable this way, so intervals are wide.

## 3. Main check: blind sample, labeled without the color field

The sheet shows the query, the product name and the start of its description. It does **not** show the color field or which
system returned the product. Fill `verdict` with `ok`, `violates` or `cant_tell`, judging the product's color from the text.

In [4]:
BLIND = "../results/blind_check.csv"
KEY = "../results/blind_check_key.csv"
N = 80
if os.path.exists(BLIND):
    print("Already exported; not overwriting.")
else:
    rows = []
    for q in color_qids:
        base, rer = runs["hybrid_rrf"][q][:K], runs["hybrid_rrf +rerank"][q][:K]
        for p in dict.fromkeys(base + rer):
            rows.append({"query_id": q, "product_id": p, "in_hybrid": p in base, "in_rerank": p in rer})
    pool = pd.DataFrame(rows)
    sample = pool.sample(min(N, len(pool)), random_state=0).reset_index(drop=True)
    sample.insert(0, "item", range(1, len(sample) + 1))
    sheet = pd.DataFrame({"item": sample["item"],
                          "query": [qtext[q] for q in sample["query_id"]],
                          "product_name": [pname[p] for p in sample["product_id"]],
                          "description_start": [pdesc[p][:300] for p in sample["product_id"]],
                          "verdict": ""})
    sheet.to_csv(BLIND, index=False)
    sample.to_csv(KEY, index=False)      # which system returned each item; do not open before labeling
    print(f"{len(pool)} distinct products in the two top-10 lists; saved {len(sheet)} to {BLIND}")

Already exported; not overwriting.


In [5]:
lab = pd.read_csv(BLIND)
key = pd.read_csv(KEY)
done = lab.merge(key, on="item").dropna(subset=["verdict"])
done = done[done["verdict"].isin(["ok", "violates", "cant_tell"])]
if len(done) < 20:
    print(f"Label at least 20 rows in {BLIND} first ({len(done)} done).")
else:
    print(f"Labeled: {len(done)}")
    res = {}
    for col, name in [("in_hybrid", "hybrid_rrf"), ("in_rerank", "hybrid_rrf +rerank")]:
        d = done[done[col]]
        decided = d[d["verdict"] != "cant_tell"]
        res[name] = {"sampled products": len(d), "decidable": len(decided),
                     "violation rate (decidable)": (decided["verdict"] == "violates").mean(),
                     "violates, share of all sampled": (d["verdict"] == "violates").mean()}
    display(pd.DataFrame(res).T.round(3))
    only_base = done[done["in_hybrid"] & ~done["in_rerank"]]
    only_rer  = done[~done["in_hybrid"] & done["in_rerank"]]
    print("Products the reranker REMOVED:", only_base["verdict"].value_counts().to_dict())
    print("Products the reranker BROUGHT IN:", only_rer["verdict"].value_counts().to_dict())

Labeled: 80


,sampled products,decidable,violation rate (decidable),"violates, share of all sampled"
hybrid_rrf,73.0,34.0,0.471,0.219
hybrid_rrf +rerank,66.0,30.0,0.433,0.197


Products the reranker REMOVED: {'cant_tell': 8, 'violates': 4, 'ok': 2}
Products the reranker BROUGHT IN: {'cant_tell': 5, 'violates': 1, 'ok': 1}


**Reading it.** These labels never used the color field, so they are independent of the reranker's signal. If the rerank list
has a lower violation rate here too, the improvement is real and not an artefact of shared data. The two counts at the end show
what the reranker swapped: removed products should be mostly `violates`, brought-in products mostly `ok`.